<a href="https://colab.research.google.com/github/hamza-24-ai/Pytorch_with_practical_deepLearning/blob/main/09_RNN_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset , DataLoader

In [7]:
df = pd.read_csv("/content/100_Unique_QA_Dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [8]:
# Tokenize

def tokenize(text):
  text = text.lower()
  text = text.replace("?","")
  text = text.replace("'","")
  return text.split()

In [9]:
# Test the tokenizer
tokenize("What is the boiling point of water in Celsius?")

['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius']

In [10]:
# Now generate a vocabulary of tokens
vocab = {'<UNK>' : 0}

In [11]:
def generate_vocab(row):
  tokenize_question = tokenize(row["question"])
  tokenize_answer = tokenize(row["answer"])

  merged_tokens = tokenize_question + tokenize_answer

  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [12]:
df.apply(generate_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [13]:
len(vocab)

324

In [14]:
# convert tokens into indices because our machine understand only numbers

def word_to_indices(text,vocab):
  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text


In [15]:
word_to_indices("What is the boiling point of water in Celsius?", vocab)

[1, 2, 3, 24, 25, 5, 26, 19, 27]

In [16]:
# create Dataset for our model

class QnA(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self,index):
    numerical_question = word_to_indices(self.df.iloc[index]["question"],self.vocab)
    numerical_answer = word_to_indices(self.df.iloc[index]["answer"],self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)


In [17]:
dataset = QnA(df,vocab)

In [18]:
# Creating dataLoader

dataLoader = DataLoader(dataset,batch_size=1, shuffle=True)

In [19]:
for question,answer in dataLoader:
  print(question,answer)

tensor([[ 42, 312,   2, 313,  62,  63,   3, 314, 315]]) tensor([[316]])
tensor([[  1,   2,   3,  33,  34,   5, 245]]) tensor([[246]])
tensor([[ 42, 299, 300, 118,  14, 301, 302, 158, 303, 304, 305, 306]]) tensor([[307]])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([[52]])
tensor([[ 1,  2,  3, 59, 25,  5, 26, 19, 60]]) tensor([[61]])
tensor([[  1,   2,   3,   4,   5, 109]]) tensor([[317]])
tensor([[10, 96,  3, 97]]) tensor([[98]])
tensor([[10,  2,  3, 66,  5, 67]]) tensor([[68]])
tensor([[ 42,   2,   3, 274, 211, 275]]) tensor([[276]])
tensor([[ 42,  18, 118,   3, 186, 187]]) tensor([[188]])
tensor([[ 1,  2,  3,  4,  5, 99]]) tensor([[100]])
tensor([[ 42, 318,   2,  62,  63,   3, 319,   5, 320]]) tensor([[321]])
tensor([[  1,   2,   3,  37,  38,  39, 161]]) tensor([[162]])
tensor([[42, 86, 87, 88, 89, 39, 90]]) tensor([[91]])
tensor([[  1,   2,   3,  92, 137,  19,   3,  45]]) tensor([[185]])
tensor([[  1,   2,   3, 163, 164, 165,  83,  84]]) tensor([[166]])
tensor([[ 1,  2,  3, 69

In [20]:
import torch.nn as nn


In [24]:
# Building Simple RNN
class MyRNN(nn.Module):

  def __init__(self,vocab_size):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50,64, batch_first=True)
    self.fc = nn.Linear(64,vocab_size)

  def forward(self, question):
    question_embed = self.embedding(question)
    hidden, final = self.rnn(question_embed)
    output = self.fc(final.squeeze(0))

    return output


In [25]:
learning_rate = 0.001
epochs = 100

In [26]:
model = MyRNN(len(vocab))

In [29]:
print(model.parameters())

<generator object Module.parameters at 0x7f6d1d3e57e0>


In [30]:
criterian = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [32]:
for epoch in range(epochs):

  total_loss = 0

  for question,answer in dataLoader:

    optimizer.zero_grad()

    output = model(question)

    loss = criterian(output, answer[0])

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch : {epoch + 1}, loss : {total_loss:.4f}")


Epoch : 1, loss : 528.9056
Epoch : 2, loss : 457.0926
Epoch : 3, loss : 376.4954
Epoch : 4, loss : 317.0027
Epoch : 5, loss : 265.8289
Epoch : 6, loss : 218.0390
Epoch : 7, loss : 173.8242
Epoch : 8, loss : 135.5667
Epoch : 9, loss : 103.2962
Epoch : 10, loss : 78.8183
Epoch : 11, loss : 60.5713
Epoch : 12, loss : 47.2935
Epoch : 13, loss : 37.3188
Epoch : 14, loss : 30.0908
Epoch : 15, loss : 24.6270
Epoch : 16, loss : 20.3740
Epoch : 17, loss : 17.1048
Epoch : 18, loss : 14.4225
Epoch : 19, loss : 12.3729
Epoch : 20, loss : 10.6787
Epoch : 21, loss : 9.3115
Epoch : 22, loss : 8.1769
Epoch : 23, loss : 7.2434
Epoch : 24, loss : 6.4768
Epoch : 25, loss : 5.8106
Epoch : 26, loss : 5.2349
Epoch : 27, loss : 4.7509
Epoch : 28, loss : 4.3112
Epoch : 29, loss : 3.9430
Epoch : 30, loss : 3.6129
Epoch : 31, loss : 3.3186
Epoch : 32, loss : 3.0535
Epoch : 33, loss : 2.8199
Epoch : 34, loss : 2.6182
Epoch : 35, loss : 2.4214
Epoch : 36, loss : 2.2502
Epoch : 37, loss : 2.0945
Epoch : 38, loss :

In [33]:
def predict(model, question, threshold=0.5):

  indexed_question = word_to_indices(question,vocab)

  # tensor
  question_tensor = torch.tensor(indexed_question).unsqueeze(0)

  # get Output

  output = model(question_tensor)

  # convert logits into probability

  prob = torch.nn.functional.softmax(output, dim=1)

  # getting max prob and index no

  probs, index = torch.max(prob, dim=1)

  # checking data
  if probs < threshold:
    print("I don't know what is answer")
  else:
    print(list(vocab.keys())[index])



In [35]:
predict(model, "What is the boiling point of water in Celsius?")

100
